In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
import glob

file_paths = glob.glob("pdf-name")
print(file_paths)

In [65]:
from llama_cloud import LlamaCloud


client = LlamaCloud(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    max_retries=3,
)


def parse_pdf(pdf_path: str, metadata_base : dict):
    
    pdf_standard = metadata_base["standard"]
    pdf_subject = metadata_base["subject"]
    pdf_chapter = metadata_base["chapter"]

    print(f"[{pdf_chapter}] Starting extraction...")
    
    pdf_file = client.files.create(file=pdf_path, purpose="parse")
    
    result = client.parsing.parse(
        file_id=pdf_file.id,
        tier="agentic_plus",
        version="latest",
        expand=["markdown"]
    )
    
    pages = result.markdown.pages
    
    full_markdown = "\n\n".join(
        page.markdown for page in result.markdown.pages
    )
    return full_markdown, metadata_base

    

In [ ]:
import re

def sanitize_ncert_markdown(raw_md):
    """Forces strict hierarchy, ignoring LlamaParse visual artifacts."""
    clean_md = raw_md
    
    # 1. Main Sections (e.g., 1.1)
    clean_md = re.sub(r'^[\s>#\*]*(\d+\.\d+\s+[A-Za-z])', r'## \1', clean_md, flags=re.MULTILINE)
    
    # 2. Sub-Sections (e.g., 1.1.1)
    clean_md = re.sub(r'^[\s>#\*]*(\d+\.\d+\.\d+\s+[A-Za-z])', r'### \1', clean_md, flags=re.MULTILINE)
    
    # 3. Activities
    clean_md = re.sub(r'^[\s>#\*]*(Activity\s+\d+\.\d+)', r'#### \1', clean_md, flags=re.MULTILINE|re.IGNORECASE)
    
    # 4. Questions
    clean_md = re.sub(r'^[\s>#\*]*(Q\s*U\s*E\s*S\s*T\s*I\s*O\s*N\s*S|QUESTIONS)', r'### QUESTIONS', clean_md, flags=re.MULTILINE|re.IGNORECASE)
    
    return clean_md

In [54]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


def split_markdown(full_markdown: str):
    headers_to_split_on = [
        ("##", "Main_Topic"),  # Splits on 1.1, 1.2, etc.
        ("###", "Sub_Topic"),  # Splits on 1.1.1, 1.2.5, etc.
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    topic_chunks = markdown_splitter.split_text(full_markdown)
    
    
    return topic_chunks

In [56]:
from langchain_core.documents import Document
def add_metadata(chunks: list[Document], pdf_metadata : dict):
    
    pdf_standard = pdf_metadata["standard"]
    pdf_subject = pdf_metadata["subject"]
    pdf_chapter = pdf_metadata["chapter"]
    
    for chunk in chunks:
        metadata = chunk.metadata.copy() 
        metadata.pop("Sub_Topic", None)
        sub_topic = "General" if metadata.get("Main_Topic", None) is None else metadata["Main_Topic"]
        metadata.pop("Main_Topic", None)
        
        metadata.update({
            "sub_topic" : sub_topic,
            "standard" : pdf_standard,
            "subject" : pdf_subject,
            "chapter" : pdf_chapter
        }) 
        
        chunk.metadata = metadata 
    
    return chunks
    

In [57]:
chapter_counter = {}
def create_db_chunks(chunks : list[Document]):
    db_chunks = []
    
    for i, chunk in enumerate(chunks):
        extracted_urls = []
        metadata = chunk.metadata
        
        chapter = metadata.get("chapter")
        standard = metadata.get("standard")
        subject = metadata.get("subject")
        sub_topic = metadata.get("sub_topic")
        
        if chapter not in chapter_counter:
            chapter_counter[chapter] = 0
        
        db_chunk = {
            "content": chunk.page_content,
            "standard" : standard,
            "subject" : subject,
            "chapter_name" : chapter,
            "sub_topic" : sub_topic,
            "chunk_index" : chapter_counter[chapter],
            "has_image" : False,
            "image_urls": extracted_urls
            
        }
        
        chapter_counter[chapter] += 1
        
        db_chunks.append(db_chunk)
        
    return db_chunks
        
    

In [58]:
from supabase import create_client, Client

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

db : Client = create_client(supabase_key=SUPABASE_KEY, supabase_url=SUPABASE_URL)



In [ ]:
from concurrent.futures import ThreadPoolExecutor

file_info = []

for file_path in file_paths:
    pdf_name = os.path.splitext(os.path.basename(file_path))[0]
    pdf_info = pdf_name.split(sep='_')

    pdf_standard = pdf_info[0]
    pdf_subject = pdf_info[1]
    pdf_chapter = pdf_info[2]
    metadata_base = {
        "standard": pdf_standard,
        "subject": pdf_subject,
        "chapter": pdf_chapter
    }
    
    file_info.append((file_path, metadata_base))


all_db_chunks = []

with ThreadPoolExecutor(max_workers=5)  as executor:
    futures = [executor.submit(parse_pdf, file_path, metadata_base) for (file_path, metadata_base) in file_info]
    
    for future in futures:
        full_markdown, metadata_base = future.result()
        clean_markdown = sanitize_ncert_markdown(raw_md=full_markdown)
        chunks = split_markdown(full_markdown=clean_markdown)
        chunks = add_metadata(chunks=chunks, pdf_metadata=metadata_base)
        db_chunks = create_db_chunks(chunks=chunks)
        all_db_chunks.extend(db_chunks)
    
    

In [ ]:
BATCH_SIZE = 50
for i in range(0, len(all_db_chunks), BATCH_SIZE):
    batch = all_db_chunks[i:i + BATCH_SIZE]

    response = db.table("chunks").insert(batch).execute()
    print(f"Uploaded batch {i // BATCH_SIZE + 1}: {len(batch)} chunks")
print(f"\nDone! Uploaded {len(all_db_chunks)} chunks total.")

